# Analisis dan Estimasi Indeks Standar Pencemar Udara (ISPU) DKI Jakarta Menggunakan Linear Regression

**Dataset:** Indeks Standar Pencemar Udara (ISPU) — Provinsi DKI Jakarta (2022–2023)  
**Sumber:** Portal Data Terbuka Indonesia — [data.go.id](https://data.go.id)  
**Algoritma:** Linear Regression  
**Target:** Estimasi nilai indeks polutan tertinggi harian (`max`)  

---

## 1. Business Understanding

### Latar Belakang
Kualitas udara di DKI Jakarta merupakan salah satu isu lingkungan paling krusial. Jakarta secara konsisten masuk daftar kota dengan polusi udara tertinggi di dunia, berdampak langsung pada kesehatan jutaan penduduk. Pemerintah DKI Jakarta memantau kualitas udara melalui 5 Stasiun Pemantau Kualitas Udara (SPKU) yang mengukur konsentrasi polutan setiap hari.

### Permasalahan
Bagaimana cara **memprediksi/mengestimasi nilai indeks polutan tertinggi harian** (`max`) berdasarkan konsentrasi masing-masing polutan (PM10, PM2.5, SO₂, CO, O₃, NO₂)? Prediksi ini berguna untuk sistem peringatan dini kualitas udara.

### Tujuan
Membangun model **Linear Regression** yang mampu mengestimasi nilai `max` (indeks polutan tertinggi harian) dari konsentrasi polutan individual, dan mengevaluasi akurasi prediksinya.

### Target Variabel
- **Input (Fitur X):** `pm_sepuluh`, `pm_duakomalima`, `sulfur_dioksida`, `karbon_monoksida`, `ozon`, `nitrogen_dioksida`
- **Output (Target Y):** `max` — nilai numerik → sesuai untuk **Linear Regression**


## 2. Import Library dan Muat Dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline

# ── Muat Dataset dari GitHub ────────────────────────────────────────────
# Ganti URL di bawah dengan raw URL CSV kamu di GitHub
# Format: https://raw.githubusercontent.com/USERNAME/REPO/main/NAMA_FILE.csv
GITHUB_CSV_URL = "https://raw.githubusercontent.com/iRust7/DataScience/main/Dataset%20Pencemaran%20Udara%20DKI%20JAKARTA.csv"

ispu_df = pd.read_csv(
    GITHUB_CSV_URL,
    encoding="utf-8-sig",
    na_values=["-", "---"],
)

print(f"Bentuk data: {ispu_df.shape}")
print(ispu_df.head())


## 3. Data Understanding

Dataset memiliki **1.825 baris** dan **12 kolom** hasil pengukuran harian dari 5 stasiun SPKU DKI Jakarta.

| Kolom | Tipe | Keterangan |
|---|---|---|
| `periode_data` | int | Kode periode YYYYMM |
| `tanggal` | object | Tanggal pengukuran (YYYY-MM-DD) |
| `stasiun` | object | Nama stasiun SPKU (5 lokasi) |
| `pm_sepuluh` | float | Konsentrasi PM10 (μg/m³) |
| `pm_duakomalima` | float | Konsentrasi PM2.5 (μg/m³) — polutan paling berbahaya |
| `sulfur_dioksida` | float | Konsentrasi SO₂ (μg/m³) |
| `karbon_monoksida` | float | Konsentrasi CO (mg/m³) |
| `ozon` | float | Konsentrasi O₃ (μg/m³) |
| `nitrogen_dioksida` | float | Konsentrasi NO₂ (μg/m³) |
| `max` | float | **Target:** Nilai indeks polutan tertinggi harian |
| `parameter_pencemar_kritis` | object | Polutan yang menghasilkan nilai max |
| `kategori` | object | BAIK / SEDANG / TIDAK SEHAT / SANGAT TIDAK SEHAT |

> **Catatan output `info()`:** Kolom polutan bertipe `float` karena nilai `-` dan `---` di CSV sudah dibaca sebagai `NaN`. Nilai hilang signifikan: PM2.5 (295 baris), PM10 (222 baris) — akan ditangani di Data Cleaning.


In [ ]:
# Struktur data
ispu_df.info()

# Statistik deskriptif
numeric_cols = ['pm_sepuluh', 'pm_duakomalima', 'sulfur_dioksida',
                'karbon_monoksida', 'ozon', 'nitrogen_dioksida', 'max']
print(ispu_df[numeric_cols].describe())

# Nilai hilang
print("\nJumlah nilai hilang per kolom:")
print(ispu_df.isna().sum())


## 4. Exploratory Data Analysis (EDA)

Tiga visualisasi untuk memahami pola data sebelum pemodelan:

1. **Distribusi `max`** — sebaran nilai indeks polutan tertinggi harian (right-skewed = mayoritas di nilai sedang, ada outlier tinggi)
2. **Kategori kualitas udara** — seberapa sering tiap kategori muncul (SEDANG diperkirakan dominan)
3. **Heatmap korelasi** — hubungan antar polutan dan target `max` (fitur dengan korelasi tinggi = prediktor kuat untuk Linear Regression)


In [ ]:
# 1. Distribusi nilai max
plt.figure(figsize=(8, 5))
sns.histplot(ispu_df['max'], bins=30, kde=True, color='steelblue')
plt.title('Distribusi Nilai Indeks Polutan Tertinggi (max)')
plt.xlabel('Nilai Indeks (max)')
plt.ylabel('Frekuensi')
plt.tight_layout()
plt.show()

# 2. Jumlah observasi per kategori
plt.figure(figsize=(7, 4))
kat_counts = ispu_df['kategori'].value_counts()
sns.barplot(x=kat_counts.index, y=kat_counts.values, palette='Blues_d')
plt.title('Jumlah Observasi per Kategori Kualitas Udara')
plt.xlabel('Kategori')
plt.ylabel('Jumlah Observasi')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# 3. Heatmap korelasi
plt.figure(figsize=(9, 6))
corr_cols = ['pm_sepuluh', 'pm_duakomalima', 'sulfur_dioksida',
             'karbon_monoksida', 'ozon', 'nitrogen_dioksida', 'max']
corr_data = ispu_df[corr_cols].apply(pd.to_numeric, errors='coerce')
sns.heatmap(corr_data.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Heatmap Korelasi Polutan terhadap Nilai Max')
plt.tight_layout()
plt.show()


## 5. Data Cleaning

| Langkah | Penjelasan | Output |
|---|---|---|
| Konversi tanggal | `object` → `datetime` | Siap ekstraksi fitur waktu |
| Hapus duplikat | `drop_duplicates()` | Hindari bias model |
| Hapus "TIDAK ADA DATA" | 21 baris tanpa pengukuran apapun | Data tidak informatif, merusak imputasi |
| Imputasi median | Isi NaN dengan median per kolom | Median lebih robust dari mean terhadap outlier |
| Strip spasi stasiun | `str.strip()` | Menyatukan nama stasiun yang sama |

> **Output yang diharapkan:** Dimensi berubah **(1.825 → 1.804 baris)** · Semua kolom: **0 nilai hilang**


In [ ]:
# Salinan data
clean_df = ispu_df.copy()

# 1. Konversi tanggal
clean_df['tanggal'] = pd.to_datetime(clean_df['tanggal'])

# 2. Hapus duplikat
clean_df = clean_df.drop_duplicates()

# 3. Hapus baris "TIDAK ADA DATA"
clean_df = clean_df[clean_df["kategori"] != "TIDAK ADA DATA"].reset_index(drop=True)

# 4. Imputasi median pada kolom numerik
num_cols = ['pm_sepuluh', 'pm_duakomalima', 'sulfur_dioksida',
            'karbon_monoksida', 'ozon', 'nitrogen_dioksida', 'max']
imputer = SimpleImputer(strategy='median')
clean_df[num_cols] = imputer.fit_transform(clean_df[num_cols])

# 5. Bersihkan spasi nama stasiun
clean_df['stasiun'] = clean_df['stasiun'].str.strip()

print(f"Dimensi data setelah cleaning: {clean_df.shape}")
print("Jumlah nilai hilang setelah cleaning:")
print(clean_df.isna().sum())


## 6. Data Preparation untuk Model

Menyiapkan fitur (X) dan target (y) untuk Linear Regression:

- **Fitur (X):** 6 kolom konsentrasi polutan — `pm_sepuluh`, `pm_duakomalima`, `sulfur_dioksida`, `karbon_monoksida`, `ozon`, `nitrogen_dioksida`
- **Target (y):** `max` — nilai indeks polutan tertinggi harian (numerik)
- **Split data:** 80% latih / 20% uji (`random_state=42` untuk reproducibility)
- **Standarisasi:** `StandardScaler` agar semua fitur berskala sama (penting untuk regresi)


In [ ]:
# Fitur dan target
feature_cols = ['pm_sepuluh', 'pm_duakomalima', 'sulfur_dioksida',
                'karbon_monoksida', 'ozon', 'nitrogen_dioksida']
X = clean_df[feature_cols]
y = clean_df['max']

# Split data 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standarisasi fitur
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Jumlah data latih : {X_train.shape[0]} baris")
print(f"Jumlah data uji   : {X_test.shape[0]} baris")
print(f"Jumlah fitur      : {X_train.shape[1]}")


## 7. Pemodelan: Linear Regression

Model **Linear Regression** dipilih karena target `max` adalah nilai **numerik kontinu**, sesuai ketentuan penggunaan Linear Regression untuk task estimasi.

Linear Regression mencari hyperplane yang meminimalkan total kuadrat error (OLS — Ordinary Least Squares):

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \beta_3 x_3 + \beta_4 x_4 + \beta_5 x_5 + \beta_6 x_6$$

di mana $x_1 \ldots x_6$ adalah konsentrasi polutan yang telah distandardisasi dan $\hat{y}$ adalah estimasi nilai `max`.


In [ ]:
# Latih model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Prediksi pada data uji
y_pred = model.predict(X_test_scaled)

# Tampilkan koefisien
coef_df = pd.DataFrame({
    'Fitur': feature_cols,
    'Koefisien': model.coef_
}).sort_values('Koefisien', ascending=False)

print("Intercept (β₀):", round(model.intercept_, 4))
print("\nKoefisien tiap fitur:")
print(coef_df.to_string(index=False))


## 8. Evaluasi Model

| Metrik | Rumus | Interpretasi |
|---|---|---|
| **MAE** | $\frac{1}{n}\sum\|y_i - \hat{y}_i\|$ | Rata-rata error absolut (satuan = satuan `max`) |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Sensitif terhadap error besar |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proporsi varians yang dijelaskan model (0–1, makin tinggi makin baik) |

**Visualisasi Actual vs Predicted:** Titik-titik mendekati garis merah diagonal = prediksi akurat. Semakin rapat ke garis, semakin baik model.


In [ ]:
# Hitung metrik
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("=" * 35)
print("       EVALUASI MODEL")
print("=" * 35)
print(f"  MAE  : {mae:.4f}")
print(f"  RMSE : {rmse:.4f}")
print(f"  R2   : {r2:.4f}")
print("=" * 35)

# Scatter plot Actual vs Predicted
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='steelblue', edgecolors='white', linewidth=0.5)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', linewidth=2, label='Prediksi Sempurna')
plt.xlabel('Nilai Aktual (max)')
plt.ylabel('Nilai Prediksi (max)')
plt.title(f'Actual vs Predicted - Linear Regression\nR² = {r2:.4f}')
plt.legend()
plt.tight_layout()
plt.show()

# Bar chart koefisien
plt.figure(figsize=(7, 4))
colors = ['tomato' if c < 0 else 'steelblue' for c in coef_df['Koefisien']]
plt.barh(coef_df['Fitur'], coef_df['Koefisien'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Nilai Koefisien')
plt.title('Kontribusi Fitur terhadap Prediksi Max')
plt.tight_layout()
plt.show()


## 9. Kesimpulan

### Ringkasan Seluruh Tahapan

| Tahap | Keterangan | Hasil |
|---|---|---|
| Business Understanding | Estimasi indeks polutan tertinggi harian | Target: `max` (numerik) → Linear Regression |
| Data Understanding | 1.825 baris, 12 kolom, banyak NaN | Dipahami struktur & kualitas data |
| EDA | Histogram, bar chart, heatmap korelasi | PM2.5 berkorelasi kuat dengan `max` |
| Data Cleaning | Imputasi median, hapus "TIDAK ADA DATA" | **1.804 baris**, 0 NaN |
| Data Preparation | Split 80:20, standarisasi fitur | Data siap latih dan uji |
| Pemodelan | Linear Regression | Koefisien tiap polutan diperoleh |
| Evaluasi | MAE, RMSE, R², visualisasi | Lihat output di atas |

### Interpretasi Hasil Model
- **R² mendekati 1** → model mampu menjelaskan sebagian besar variasi nilai `max` dari konsentrasi polutan
- **Koefisien terbesar** → polutan dengan koefisien positif terbesar adalah kontributor utama kenaikan nilai `max`
- **MAE** menunjukkan rata-rata selisih prediksi dengan nilai aktual dalam satuan indeks ISPU

### Kesimpulan Utama
1. **PM2.5 adalah prediktor terkuat** — korelasi tertinggi dengan `max` dan koefisien regresi terbesar
2. **Linear Regression terbukti efektif** untuk task estimasi nilai indeks polutan tertinggi harian
3. **Kategori SEDANG mendominasi** (>60% hari) di DKI Jakarta 2022–2023, menunjukkan kondisi udara yang belum memuaskan
4. Model ini dapat dikembangkan dengan fitur tambahan (data cuaca, hari kerja/libur) untuk meningkatkan akurasi
